# FinBERT Sentiment Scoring — Phase 1

**Academic research only — not investment advice.** This project compares model-derived news sentiment against market features for S&P 500 sector ETFs; nothing here is a trading signal.

Scores each distinct headline (prepared in `07a_llm_scoring_input_prep.ipynb`) once with **FinBERT** (`ProsusAI/finbert`), a transformer fine-tuned on financial text, to produce a sentiment score compared head-to-head against RavenPack's built-in `event_sentiment_score`.

**Why FinBERT (local, free).** Runs entirely on this machine, so the WRDS-licensed headlines never leave it (license-safe) and there is no API cost. It is the "specialised financial transformer" thread from the project's own literature review. This makes the comparison **FinBERT vs RavenPack** — two purpose-built financial sentiment scorers.

**Functional-separation firewall (the leakage defense).** FinBERT is shown *only the headline text* — never prices, dates, tickers, or what happened afterward — and only rates the tone of that text. Directional prediction is done downstream by the objective walk-forward classifier (Phase 2), never here. Because the model cannot see outcomes or dates, its score cannot be contaminated by knowledge of what the market did.

**Score mapping.** FinBERT returns probabilities for positive / negative / neutral. We derive:
`sentiment = P(positive) − P(negative)` (a −1..+1 signal, directly comparable to RavenPack), `label = argmax`, `confidence = max probability`.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# --- paths (work whether run from repo root or data_collection/) --------------
CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "data_collection").exists() else CWD.parent
RAW_DIR = REPO_ROOT / "data_collection" / "raw"

SCORING_INPUT_CSV = RAW_DIR / "llm_scoring_input.csv"   # from 07a (distinct texts)
SCORES_CSV = RAW_DIR / "finbert_scores.csv"             # this notebook writes here

MODEL_NAME = "ProsusAI/finbert"
INFER_BATCH = 64        # texts per forward pass
SAVE_EVERY = 20_000     # append to disk this often (resume safety)
MAX_LEN = 128           # headlines are short; 128 tokens is ample

# GPU if a CUDA build of torch is installed. The default pip torch is CPU-only;
# to use an NVIDIA GPU, reinstall a CUDA build, e.g.:
#   pip install --index-url https://download.pytorch.org/whl/cu128 torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cpu":
    print("  (running on CPU — works, just slower. See the pip command above for GPU.)")
print(f"scoring input: {SCORING_INPUT_CSV} (exists: {SCORING_INPUT_CSV.exists()})")

## 1. Load FinBERT

Downloads `ProsusAI/finbert` from Hugging Face on first run (~440 MB, public — no token needed) and caches it. We read the label order from the model config rather than hard-coding indices, so the positive/negative/neutral mapping is always correct.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE).eval()

# id2label from the model config, lowercased -> {0:'positive',1:'negative',2:'neutral'}
ID2LABEL = {i: lbl.lower() for i, lbl in model.config.id2label.items()}
LABELS = [ID2LABEL[i] for i in range(len(ID2LABEL))]
print("label order:", LABELS)
assert set(LABELS) == {"positive", "negative", "neutral"}, LABELS

## 2. Scoring function

Tokenize a batch of headlines, run one forward pass, softmax to probabilities, then map to `label` / `sentiment` / `confidence`. `torch.no_grad()` keeps memory low and inference fast.

In [ ]:
@torch.no_grad()
def score_batch(texts: list) -> list:
    """Return [{label, sentiment, confidence}, ...] for a list of headline strings."""
    enc = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN
    ).to(DEVICE)
    probs = F.softmax(model(**enc).logits, dim=-1).cpu()  # (n, 3)

    pos_i = LABELS.index("positive")
    neg_i = LABELS.index("negative")
    out = []
    for row in probs:
        p = {LABELS[i]: float(row[i]) for i in range(len(LABELS))}
        top = max(p, key=p.get)
        out.append({
            "label": top,
            "sentiment": p["positive"] - p["negative"],  # -1..+1, comparable to RavenPack
            "confidence": row.max().item(),
        })
    return out

## 3. Sample run — sanity-check the scores

Score the first few headlines and eyeball that the labels and signs look right before committing to the full run.

In [ ]:
SAMPLE_SIZE = 12
scoring_input = pd.read_csv(SCORING_INPUT_CSV)

sample = scoring_input.head(SAMPLE_SIZE)
sample_scores = pd.DataFrame(score_batch(sample["headline"].tolist()))
sample_scores.insert(0, "headline", sample["headline"].values)
pd.set_option("display.max_colwidth", 70)
sample_scores

## 4. Full run — score every distinct headline (resumable)

Scores all texts not already in `finbert_scores.csv`, in GPU/CPU batches, appending to disk every `SAVE_EVERY` rows so an interruption loses at most that many. Re-running resumes where it left off. This is local and free — no rate limits, no cost.

In [ ]:
SCORE_COLS = ["text_id", "label", "sentiment", "confidence", "model", "scored_at"]


def done_text_ids() -> set:
    if SCORES_CSV.exists():
        return set(pd.read_csv(SCORES_CSV, usecols=["text_id"])["text_id"])
    return set()


def append_scores(rows: list) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows, columns=SCORE_COLS)
    df.to_csv(SCORES_CSV, mode="a", header=not SCORES_CSV.exists(), index=False)


todo = scoring_input[~scoring_input["text_id"].isin(done_text_ids())].reset_index(drop=True)
print(f"texts to score: {len(todo):,}")

buffer, t0 = [], time.time()
for start in range(0, len(todo), INFER_BATCH):
    chunk = todo.iloc[start:start + INFER_BATCH]
    scores = score_batch(chunk["headline"].tolist())
    now = pd.Timestamp.utcnow().isoformat()
    for tid, s in zip(chunk["text_id"], scores):
        buffer.append({"text_id": tid, **s, "model": MODEL_NAME, "scored_at": now})
    if len(buffer) >= SAVE_EVERY:
        append_scores(buffer)
        done = start + len(chunk)
        print(f"  {done:,}/{len(todo):,} scored ({done/(time.time()-t0):.0f}/s)")
        buffer = []
append_scores(buffer)
print(f"done in {time.time()-t0:.1f}s")

## 5. Validate and hand off to Phase 2

Check coverage and the score distribution. Phase 2 then joins these scores back to every event via `llm_scoring_event_map.csv` (on `text_id`) and aggregates to the trading-session grain using the same 4 PM ET cutoff rules as the existing pipeline, so the FinBERT panel lines up 1:1 with the RavenPack panel.

In [ ]:
scores = pd.read_csv(SCORES_CSV) if SCORES_CSV.exists() else pd.DataFrame(columns=SCORE_COLS)
n_input = len(scoring_input)
print(f"distinct texts:  {n_input:,}")
print(f"scored:          {len(scores):,}  ({len(scores)/n_input:.1%} coverage)")
print(f"duplicate text_ids in scores: {scores['text_id'].duplicated().sum()}")
print()
if len(scores):
    print("label distribution:")
    print(scores["label"].value_counts())
    print()
    print("sentiment summary (P(pos) - P(neg)):")
    print(scores["sentiment"].describe()[["mean", "std", "min", "max"]])
    display(scores.head())

---

**Next — Phase 2 (comparison):**
- **Score-level:** merge `finbert_scores.csv` with `llm_scoring_event_map.csv` on `text_id` to attach RavenPack's `event_sentiment_score`; measure agreement (correlation, confusion matrix on the three labels) and characterize *where* FinBERT and RavenPack disagree — this produces findings regardless of predictive power.
- **Model-level:** aggregate FinBERT scores to sessions (reuse the 4 PM cutoff / session-mapping logic), then re-run the walk-forward harness from `06_xlk_baseline_model.ipynb` with three feature sets — market-only, market+RavenPack, market+FinBERT — on identical folds and metrics.
- **Model-free sanity check:** forward returns stratified by FinBERT sentiment bucket.